# BART save → load → predict

Read [README.md](README.md) for installation, the API contract and limitations. Run in an environment with `.[current,notebook]` or `.[legacy,notebook]` installed.

In [ ]:
from pathlib import Path
import tempfile
import subprocess
import sys
import numpy as np
import pymc as pm
import pymc_bart as pmb
from bart_persistence.save import save_bart
from bart_persistence.load import load_bart

Path("artifacts").mkdir(exist_ok=True)
folder = Path(tempfile.mkdtemp(prefix="walkthrough-", dir="artifacts")).resolve()
folder

## 1. Train a small example

In [ ]:
rng = np.random.default_rng(42)
X = rng.normal(size=(60, 2))
y = np.sin(X[:, 0]) + 0.5 * X[:, 1] + rng.normal(0, 0.1, 60)
with pm.Model() as model:
    x = pm.Data("X", X)
    mu = pmb.BART("mu", x, y, m=10)
    pm.Normal("observed", mu, 0.1, observed=y)
    trace = pm.sample(draws=30, tune=30, chains=1, cores=1,
                      random_seed=42, progressbar=False,
                      compute_convergence_checks=False)

## 2. Save the BART variable

In [ ]:
path = folder / "model.bart"
save_bart(model["mu"], path,
          expected_draws=trace.posterior.sizes["chain"] * trace.posterior.sizes["draw"])
print(f"Saved {path.stat().st_size:,} bytes to {path}")

## 3. Load a reusable predictor and predict new rows

In [ ]:
X_new = rng.normal(size=(7, 2))
predictor = load_bart(path)
samples = predictor.predict(X_new, draws=100, seed=7)
assert samples.shape == (100, 7)
print("Retained ensembles:", predictor.n_draws)
print("Latent posterior mean:", samples.mean(axis=0))
print("95% latent interval:", np.quantile(samples, [0.025, 0.975], axis=0))

## 4. Check against the live upstream trees

In [ ]:
import importlib.metadata as md
from pymc_bart.utils import _sample_posterior
if md.version("pymc-bart") == "0.13.1":
    from pymc_bart.utils import _get_posterior_sampler
    original = _get_posterior_sampler(model["mu"].owner.op)
else:
    original = list(model["mu"].owner.op.all_trees)
expected = _sample_posterior(original, X_new, np.random.default_rng(7), size=100)[:, :, 0]
np.testing.assert_array_equal(samples, expected)
print("All predictions match the original trained trees exactly.")

## 5. Verify loading in a fresh Python process

In [ ]:
np.save(folder / "X_new.npy", X_new)
np.save(folder / "expected.npy", expected)
script = """
import numpy as np
import pymc as pm
from bart_persistence.load import load_bart

def forbidden(*args, **kwargs):
    raise AssertionError("Prediction must not train or rebuild a model")
pm.sample = forbidden
pm.Model = forbidden
predictor = load_bart("model.bart")
actual = predictor.predict(np.load("X_new.npy"), draws=100, seed=7)
np.testing.assert_array_equal(actual, np.load("expected.npy"))
print("Fresh process: exact agreement without retraining or a rebuilt model.")
"""
result = subprocess.run([sys.executable, "-c", script], cwd=folder,
                        capture_output=True, text=True, timeout=120)
assert result.returncode == 0, result.stderr
print(result.stdout)